In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

BASE_PATH = "/content/drive/MyDrive/automated-data-processing-reporting"

print(os.listdir(BASE_PATH))

['data']


In [ ]:
DATA_PATH = BASE_PATH + "/data"

print(os.listdir(DATA_PATH))

['raw', 'processed']


In [ ]:
RAW_PATH = DATA_PATH + "/raw"

print(os.listdir(RAW_PATH))

['categories.csv', 'customers.csv', 'employees.csv', 'order_items.csv', 'orders.csv', 'payments.csv', 'products.csv', 'promotions.csv', 'returns.csv', 'shipments.csv', 'stores.csv', 'suppliers.csv']


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

data = {}

for file in os.listdir(RAW_PATH):
    if file.endswith(".csv"):
        name = file.replace(".csv", "")
        data[name] = pd.read_csv(os.path.join(RAW_PATH, file))

print(data.keys())

dict_keys(['categories', 'customers', 'employees', 'order_items', 'orders', 'payments', 'products', 'promotions', 'returns', 'shipments', 'stores', 'suppliers'])


In [ ]:
orders = data['orders']
order_items = data['order_items']

sales_fact = order_items.merge(
    orders,
    on='order_id',
    how='left'
)

print(sales_fact.shape)
sales_fact.head()

(600000, 9)


,order_item_id,order_id,product_id,qty,price,customer_id,store_id,order_date,promotion_id
0,1,145042,472,3,176,25753,40,2020-11-07,45
1,2,110932,1666,2,1034,34919,30,2022-03-03,6
2,3,269799,8616,4,2290,29401,73,2022-05-30,20
3,4,298741,9909,3,1555,7339,61,2021-08-05,32
4,5,105218,1179,3,936,13027,16,2022-11-01,19


In [ ]:
products = data['products']

sales_fact = sales_fact.merge(
    products,
    on='product_id',
    how='left'
)

print(sales_fact.shape)
sales_fact.head()

(600000, 12)


,order_item_id,order_id,product_id,qty,price_x,customer_id,store_id,order_date,promotion_id,category_id,supplier_id,price_y
0,1,145042,472,3,176,25753,40,2020-11-07,45,13,189,4048
1,2,110932,1666,2,1034,34919,30,2022-03-03,6,2,127,4957
2,3,269799,8616,4,2290,29401,73,2022-05-30,20,18,27,2728
3,4,298741,9909,3,1555,7339,61,2021-08-05,32,19,126,353
4,5,105218,1179,3,936,13027,16,2022-11-01,19,9,84,4417


In [ ]:
suppliers = data["suppliers"]

sales_fact = sales_fact.merge(
    suppliers,
    on="supplier_id",
    how="left"
)

print(sales_fact.shape)

(600000, 16)


In [ ]:
customers = data["customers"]
stores = data["stores"]

sales_fact = sales_fact.merge(
    stores,
    on="store_id",
    how="left"
)


sales_fact = sales_fact.merge(
    customers,
    on="customer_id",
    how="left"
)

print(sales_fact.shape)

(600000, 19)


In [ ]:
promotion = data['promotions']

sales_fact = sales_fact.merge(
    promotion,
    on='promotion_id',
    how='left'
)

print(sales_fact.shape)


(600000, 20)


In [ ]:
# Fresh sales_fact

sales_fact = data["order_items"].merge(
    data["orders"],
    on="order_id",
    how="left"
)

sales_fact = sales_fact.merge(
    data["products"],
    on="product_id",
    how="left"
)

sales_fact = sales_fact.merge(
    data["categories"],
    on="category_id",
    how="left"
)

sales_fact = sales_fact.merge(
    data["suppliers"],
    on="supplier_id",
    how="left"
)

sales_fact = sales_fact.merge(
    data["customers"][["customer_id", "city"]].rename(
        columns={"city": "city_customer"}
    ),
    on="customer_id",
    how="left"
)

sales_fact = sales_fact.merge(
    data["stores"][["store_id", "city"]].rename(
        columns={"city": "city_store"}
    ),
    on="store_id",
    how="left"
)

sales_fact = sales_fact.merge(
    data["promotions"],
    on="promotion_id",
    how="left"
)

print(sales_fact.shape)
print(sales_fact.columns.tolist())

(600000, 17)
['order_item_id', 'order_id', 'product_id', 'qty', 'price_x', 'customer_id', 'store_id', 'order_date', 'promotion_id', 'category_id', 'supplier_id', 'price_y', 'category_name', 'country', 'city_customer', 'city_store', 'discount']


In [ ]:
print(sales_fact.columns.tolist())

['order_item_id', 'order_id', 'product_id', 'qty', 'price_x', 'customer_id', 'store_id', 'order_date', 'promotion_id', 'category_id', 'supplier_id', 'price_y', 'category_name_x', 'category_name_y', 'category_name', 'country', 'city_x', 'city_y', 'signup_date', 'discount']


In [ ]:
sales_fact["gross_revenue"] = (
    sales_fact["qty"] * sales_fact["price_x"]
)

sales_fact["discount_amount"] = (
    sales_fact["gross_revenue"] * sales_fact["discount"] / 100
)

sales_fact["net_revenue"] = (
    sales_fact["gross_revenue"] - sales_fact["discount_amount"]
)

sales_fact[
    [
        "qty",
        "price_x",
        "discount",
        "gross_revenue",
        "discount_amount",
        "net_revenue"
    ]
].head()

,qty,price_x,discount,gross_revenue,discount_amount,net_revenue
0,3,176,38,528,200.64,327.36
1,2,1034,10,2068,206.80,1861.20
2,4,2290,16,9160,1465.60,7694.40
3,3,1555,34,4665,1586.10,3078.90
4,3,936,37,2808,1038.96,1769.04


In [ ]:
print(sales_fact.columns.tolist())

['order_item_id', 'order_id', 'product_id', 'qty', 'price_x', 'customer_id', 'store_id', 'order_date', 'promotion_id', 'category_id', 'supplier_id', 'price_y', 'category_name', 'country', 'city_customer', 'city_store', 'discount']


In [ ]:
returns = data["returns"]

returns_summary = (
    returns.groupby("order_item_id")
    .agg(
        return_count=("return_id", "count"),
        refund_amount=("refund", "sum")
    )
    .reset_index()
)

print(returns_summary.head())

   order_item_id  return_count  refund_amount
0             17             1           2022
1             74             1           2621
2             89             1           4222
3             97             1           1200
4            101             1           2504


In [ ]:
sales_fact = sales_fact.merge(
    returns_summary,
    on="order_item_id",
    how="left"
)

sales_fact["return_count"] = sales_fact["return_count"].fillna(0)
sales_fact["refund_amount"] = sales_fact["refund_amount"].fillna(0)

sales_fact["is_returned"] = (
    sales_fact["return_count"] > 0
).astype(int)

print(sales_fact.shape)

(600000, 23)


In [ ]:
returns_summary = (
    data["returns"]
    .groupby("order_item_id")
    .agg(
        return_count=("return_id", "count"),
        refund_amount=("refund", "sum")
    )
    .reset_index()
)

print(returns_summary.head())

   order_item_id  return_count  refund_amount
0             17             1           2022
1             74             1           2621
2             89             1           4222
3             97             1           1200
4            101             1           2504


In [ ]:
sales_fact = sales_fact.merge(
    returns_summary,
    on="order_item_id",
    how="left"
)

sales_fact["return_count"] = (
    sales_fact["return_count"].fillna(0)
)

sales_fact["refund_amount"] = (
    sales_fact["refund_amount"].fillna(0)
)

sales_fact["is_returned"] = (
    sales_fact["return_count"] > 0
).astype(int)

print(sales_fact.shape)

(600000, 27)


In [ ]:
sales_fact["net_revenue_after_refund"] = (
    sales_fact["net_revenue"]
    - sales_fact["refund_amount"]
)

sales_fact[
    [
        "net_revenue",
        "refund_amount",
        "net_revenue_after_refund"
    ]
].head()

,net_revenue,refund_amount,net_revenue_after_refund
0,327.36,0.0,327.36
1,1861.20,0.0,1861.20
2,7694.40,0.0,7694.40
3,3078.90,0.0,3078.90
4,1769.04,0.0,1769.04


In [ ]:
sales_fact = sales_fact[
    [
        "order_id",
        "order_item_id",
        "order_date",
        "customer_id",
        "store_id",
        "product_id",
        "category_id",
        "supplier_id",
        "promotion_id",
        "city_customer",
        "city_store",
        "qty",
        "price_x",
        "discount",
        "gross_revenue",
        "discount_amount",
        "net_revenue",
        "return_count",
        "refund_amount",
        "is_returned",
        "net_revenue_after_refund"
    ]
]

In [ ]:
sales_fact = sales_fact.rename(
    columns={"price_x": "price"}
)

print(sales_fact.columns.tolist())

['order_id', 'order_item_id', 'order_date', 'customer_id', 'store_id', 'product_id', 'category_id', 'supplier_id', 'promotion_id', 'city_customer', 'city_store', 'qty', 'price', 'discount', 'gross_revenue', 'discount_amount', 'net_revenue', 'return_count', 'refund_amount', 'is_returned', 'net_revenue_after_refund']


In [ ]:
print("Shape:", sales_fact.shape)

print("\nMissing values:")
print(sales_fact.isnull().sum())

print("\nDuplicate rows:")
print(sales_fact.duplicated().sum())

Shape: (600000, 21)

Missing values:
order_id                    0
order_item_id               0
order_date                  0
customer_id                 0
store_id                    0
product_id                  0
category_id                 0
supplier_id                 0
promotion_id                0
city_customer               0
city_store                  0
qty                         0
price                       0
discount                    0
gross_revenue               0
discount_amount             0
net_revenue                 0
return_count                0
refund_amount               0
is_returned                 0
net_revenue_after_refund    0
dtype: int64

Duplicate rows:
0


In [ ]:
output_file = PROCESSED_PATH / "sales_fact.csv"

sales_fact.to_csv(
    output_file,
    index=False
)

print("Saved successfully!")
print(output_file)

Saved successfully!
/content/drive/MyDrive/automated-data-processing-reporting/data/processed/sales_fact.csv


In [ ]:
from pathlib import Path

PROCESSED_PATH = Path(
    "/content/drive/MyDrive/automated-data-processing-reporting/data/processed"
)

PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

output_file = PROCESSED_PATH / "sales_fact.csv"

sales_fact.to_csv(output_file, index=False)

print("Saved successfully!")
print(output_file)

Saved successfully!
/content/drive/MyDrive/automated-data-processing-reporting/data/processed/sales_fact.csv


In [ ]:
print(output_file.exists())
print(os.listdir(PROCESSED_PATH))

True
['sales_fact.csv']


In [ ]:
from google.colab import files

files.download(str(output_file))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>